# AWS DynamoDB CRUD Operations with Python (Boto3)

A beginner-friendly Jupyter Notebook to **Create, Read, Update, and Delete** student records in AWS DynamoDB.

## Before you start

1. Create an AWS account (if you do not already have one).
2. In the AWS Console, open **DynamoDB → Tables → Create table**.
3. Create a table with:
   - **Table name:** `students`
   - **Partition key:** `student_id`
   - **Partition key type:** `String`
4. Choose the AWS Region you will use (this notebook uses `ap-south-1`, Mumbai).
5. Configure AWS credentials securely. Do not paste access keys into this notebook.

## 1. Install Boto3

Boto3 is the AWS SDK for Python. It lets Python communicate with AWS services.

In [1]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


## 2. Configure AWS credentials

**Recommended for a computer with AWS CLI installed:** run `aws configure` in your terminal (not in a Python cell), then enter your IAM access key, secret key, and region.

Use an IAM identity that has permission to access the `students` DynamoDB table. On AWS compute such as EC2, prefer an attached IAM role instead of access keys.

Example region: `ap-south-1`. Never share or hardcode your secret access key.

In [4]:
import boto3

dynamodb = boto3.resource(
    "dynamodb",
    region_name="us-east-1"
)

table = dynamodb.create_table(
    TableName="students",
    KeySchema=[
        {
            "AttributeName": "student_id",
            "KeyType": "HASH"
        }
    ],
    AttributeDefinitions=[
        {
            "AttributeName": "student_id",
            "AttributeType": "S"
        }
    ],
    BillingMode="PAY_PER_REQUEST"
)

print("Table created successfully!")

Table created successfully!


In [6]:
import boto3

REGION = "us-east-1"
TABLE_NAME = "students"

dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

# Check that AWS credentials and table access work
print("Table:", table.table_name)
print("Status:", table.table_status)

Table: students
Status: ACTIVE


## 3. CREATE – Add a student

`put_item()` adds an item to DynamoDB. Each item must include the partition key `student_id`.

Run this cell to add a sample student. If the same `student_id` already exists, `put_item()` replaces that item.

In [7]:
table.put_item(
    Item={
        "student_id": "101",
        "name": "Veera",
        "age": 30,
        "course": "Python"
    }
)

print("Student added!")

Student added!


## 4. READ – Get one student

`get_item()` retrieves one item using its primary key.

In [8]:
response = table.get_item(
    Key={"student_id": "101"}
)

student = response.get("Item")

if student:
    print(student)
else:
    print("Student not found")

{'student_id': '101', 'name': 'Veera', 'course': 'Python', 'age': Decimal('30')}


## 5. READ – Get all students

`scan()` reads items from the table. It is convenient for a small practice table, but scans can be slower and more expensive as a table grows.

In [9]:
response = table.scan()

for student in response["Items"]:
    print(student)

{'student_id': '101', 'name': 'Veera', 'course': 'Python', 'age': Decimal('30')}


## 6. UPDATE – Change a student's age

`update_item()` finds an item using its primary key and changes the requested field.

- `Key`: identifies the item.
- `UpdateExpression`: says what to change.
- `ExpressionAttributeValues`: supplies the new value.

In [10]:
table.update_item(
    Key={"student_id": "101"},
    UpdateExpression="SET age = :new_age",
    ExpressionAttributeValues={":new_age": 31}
)

print("Student updated!")

Student updated!


### Check the updated student

In [11]:
response = table.get_item(
    Key={"student_id": "101"}
)

print(response.get("Item", "Student not found"))

{'age': Decimal('31'), 'course': 'Python', 'name': 'Veera', 'student_id': '101'}


## 7. DELETE – Remove a student

`delete_item()` deletes an item using its primary key. This deletion is permanent.

In [12]:
table.delete_item(
    Key={"student_id": "101"}
)

print("Student deleted!")

Student deleted!


### Confirm the student was deleted

In [13]:
response = table.get_item(
    Key={"student_id": "101"}
)

print(response.get("Item", "Student not found"))

Student not found


## 8. Quick revision

| CRUD operation | Boto3 method | Purpose |
|---|---|---|
| Create | `put_item()` | Add or replace an item |
| Read one | `get_item()` | Get one item by primary key |
| Read all | `scan()` | Read items in the table |
| Update | `update_item()` | Change item attributes |
| Delete | `delete_item()` | Remove an item |

## 9. Practice yourself

Try these tasks by changing the sample values:

1. Add a student with ID `102`, name `Sai`, age `25`, and course `AWS`.
2. Get only student `102`.
3. Display all students.
4. Change Sai's age to `26`.
5. Delete student `102`.

**Common errors**
- `ResourceNotFoundException`: check that the table exists and the table name is exactly `students`.
- `AccessDeniedException`: check that your IAM user or role has the required DynamoDB permissions.
- Credential error: configure credentials with `aws configure` or use an attached IAM role.
- Region error: make sure the table and `REGION` are in the same AWS Region.